<a href="https://colab.research.google.com/github/BuruhArloji/PythonDataScienceHandbook/blob/master/MT_Weekly_Forecast_2026_08_17_2026_08_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Demand Forecasting System - Modern Trade**

Notebook ini digunakan untuk menghasilkan prediksi penjualan mingguan secara otomatis dengan mempertimbangkan **riwayat penjualan**, **pembersihan data retur (NCC)**, dan **uplift promosi**.

## Alur Kerja Utama (Pipeline Overview)
1.  **Data Loading**: Mengambil data transaksi mentah dan kalender promo dari Google Drive.
2.  **NCC Cleaning**: Membersihkan data dari pembatalan transaksi (*cancellation*) menggunakan logika LIFO.
3.  **De-promotion**: Menghitung *baseline* penjualan asli dengan cara 'menghapus' lonjakan yang disebabkan oleh promo masa lalu.
4.  **Demand Classification**: Mengelompokkan setiap produk ke dalam kategori *Stable, Volatile, Intermittent,* atau *Sparse*.
5.  **Model Selection**: Melakukan *backtesting* otomatis untuk memilih algoritma terbaik (Naive, Moving Average, atau Seasonal) per SKU.
6.  **Uplift Calculation**: Menghitung seberapa besar pengaruh setiap jenis promo terhadap kenaikan volume.
7.  **Final Forecast**: Menggabungkan baseline masa depan dengan rencana promo mendatang untuk menghasilkan angka estimasi total.

---
## **Persiapan Lingkungan (Setup)**
Langkah awal adalah menginstal *library* yang dibutuhkan dan menghubungkan notebook ke penyimpanan Google Drive.

# Modern Trade Weekly Forecast — Daily to Weekly

Notebook end-to-end untuk forecast minggu 17–23 dan 24–30 Agustus 2026. Grain kerja: `DATE × Group × WH × Item Code`; output: `WEEK START × WH × Item Code`.


## 1. Configuration and reusable Python functions
Seluruh fungsi disimpan di file Python pendamping agar notebook tetap mudah diaudit dan dijalankan ulang.


In [24]:
!pip install -q pandas numpy openpyxl xlsxwriter plotly

In [25]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
from pathlib import Path
import sys
import importlib

BASE_DIR = Path("/content/drive/MyDrive/MT_Forecast")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_FILE = BASE_DIR / "MT_Weekly_Forecast_Pipeline.py"

SALES_FILE = INPUT_DIR / "Indonesia Sales Dashboard 2026.xlsm"
MASTER_FILE = INPUT_DIR / "Master_Internal.xlsx"
PROMO_FILE = INPUT_DIR / "MT Promo Calendar Form.xlsx"

assert BASE_DIR.exists(), f"Folder utama tidak ditemukan: {BASE_DIR}"
assert PIPELINE_FILE.exists(), f"Pipeline tidak ditemukan: {PIPELINE_FILE}"
assert SALES_FILE.exists(), f"Sales tidak ditemukan: {SALES_FILE}"
assert MASTER_FILE.exists(), f"Master tidak ditemukan: {MASTER_FILE}"
assert PROMO_FILE.exists(), f"Promo tidak ditemukan: {PROMO_FILE}"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import MT_Weekly_Forecast_Pipeline as fp
importlib.reload(fp)

fp.OUTPUT_DIR = OUTPUT_DIR
fp.OUTPUT_XLSX = (
    OUTPUT_DIR / "MT_Weekly_Forecast_2026-08-17_2026-08-24.xlsx"
)
fp.OUTPUT_HTML = (
    OUTPUT_DIR / "MT_Weekly_Forecast_2026-08-17_2026-08-24_Plotly.html"
)

print("File ditemukan:")
print("Sales   :", SALES_FILE)
print("Master  :", MASTER_FILE)
print("Promo   :", PROMO_FILE)
print("Pipeline:", PIPELINE_FILE)
print("Output  :", OUTPUT_DIR)

File ditemukan:
Sales   : /content/drive/MyDrive/MT_Forecast/input/Indonesia Sales Dashboard 2026.xlsm
Master  : /content/drive/MyDrive/MT_Forecast/input/Master_Internal.xlsx
Promo   : /content/drive/MyDrive/MT_Forecast/input/MT Promo Calendar Form.xlsx
Pipeline: /content/drive/MyDrive/MT_Forecast/MT_Weekly_Forecast_Pipeline.py
Output  : /content/drive/MyDrive/MT_Forecast/output


## 2. Load and standardize 2025–2026 sales


In [27]:
raw = fp.load_sales_sources(SALES_FILE)

raw.groupby("SOURCE_SHEET").agg(
    rows=("ROW_ID", "size"),
    min_date=("DATE", "min"),
    max_date=("DATE", "max"),
    qty=("QTY_PACK", "sum"),
)

,rows,min_date,max_date,qty
SOURCE_SHEET,,,,
Sales 2025,20354,2025-01-02,2025-12-31,2839537.0
Sales 2026,18264,2026-01-02,2026-08-12,1770782.0


In [28]:
import pandas as pd
import importlib

# Memuat ulang data dari file untuk mendeteksi perubahan terbaru
importlib.reload(fp)
raw = fp.load_sales_sources(SALES_FILE)

# Memeriksa data transaksi terakhir di sheet Sales 2026
sales_2026 = raw[raw['SOURCE_SHEET'] == 'Sales 2026']
max_date = sales_2026['DATE'].max()
latest_transactions = sales_2026.sort_values('DATE', ascending=False).head(15)

print(f"HASIL PENGECEKAN TERBARU:")
print(f"Tanggal transaksi paling akhir yang terdeteksi: {max_date}")

if max_date >= pd.Timestamp('2026-08-12'):
    print("\n✅ Sukses! Data hingga 12 Agustus atau lebih baru sudah terdeteksi.")
else:
    print(f"\n⚠️ Perhatian: Data masih mentok di tanggal {max_date}. Pastikan file sudah di-save dan sinkronisasi Google Drive selesai.")

print("\n15 Transaksi Terakhir di Data Mentah (Raw):")
display(latest_transactions[['DATE', 'CUSTOMER_NAME', 'WH', 'ITEM_CODE', 'QTY_PACK']])

Tanggal transaksi paling akhir yang terdeteksi: 2026-08-12 00:00:00

15 Transaksi Terakhir di Data Mentah (Raw):


,DATE,CUSTOMER_NAME,WH,ITEM_CODE,QTY_PACK
38652,2026-08-12,PT.MAHAWIRYA MAKMUR SENTOSA,LOGISTIC PROVIDER BEKASI,500040,21.0
38545,2026-08-12,LOTTE MART_CROSS DOCK,FACTORY,500040,1.0
38552,2026-08-12,ASIA TOSERBA_CIREBON,FACTORY,500040,5.0
38551,2026-08-12,PT. SUMBER ALFARIA TRIJAYA WH BENGKULU,FACTORY,523506,1551.0
38550,2026-08-12,PT. SUMBER ALFARIA TRIJAYA WH BENGKULU,FACTORY,523506,622.0
38549,2026-08-12,PT. SUMBER ALFARIA TRIJAYA LUWU,FACTORY,523506,-1.0
38548,2026-08-12,LOTTE MART_CROSS DOCK,FACTORY,523805,25.0
38547,2026-08-12,LOTTE MART_CROSS DOCK,FACTORY,523414,1.0
38546,2026-08-12,LOTTE MART_CROSS DOCK,FACTORY,523410,30.0
38544,2026-08-12,PT LOTTE SHOPPING INDONESIA_X-DOCK,FACTORY,523805,70.0


## 3. NCC cleaning
Exact LIFO menggunakan WH + customer + item + price, lalu residual dicari pada pembelian customer yang sama dalam 30 hari sebelumnya.


In [31]:
clean, ncc_allocations, ncc_unmatched, ncc_qc = fp.clean_ncc(raw)
ncc_qc


{'source_valid_rows': 38618,
 'cancellation_rows': 1964,
 'cancellation_qty': 95569.0,
 'first_pass_matched_qty': 94352.0,
 'second_pass_matched_qty': 458.0,
 'final_unmatched_qty': 759.0,
 'clean_rows_all_channels': 35731,
 'clean_qty_all_channels': 4611078.0}

## 4. Filter Modern Trade and map official Group


In [32]:
customer_master = fp.load_customer_master(MASTER_FILE)

mt, customer_unmatched = fp.map_modern_trade(
    clean,
    customer_master
)

product_dim = fp.build_product_dim(mt)

{
    "rows": len(mt),
    "packs": mt[fp.QTY].sum(),
    "groups": mt["GROUP"].nunique(),
    "wh": mt["WH"].nunique(),
    "sku": mt["ITEM_CODE"].nunique(),
    "unmatched_customer": len(customer_unmatched),
}

{'rows': 26068,
 'packs': np.float64(3838286.0),
 'groups': 30,
 'wh': 11,
 'sku': 30,
 'unmatched_customer': 0}

### Sampel Data Bersih Transaksi Modern Trade
Berikut adalah tampilan data penjualan setelah dibersihkan dari NCC dan difilter khusus untuk segmen Modern Trade.

In [33]:
# Menampilkan 5 baris pertama data transaksi MT yang sudah bersih
display(mt[['DATE', 'GROUP', 'WH', 'ITEM_CODE', 'SKU_DESCRIPTION', 'QTY_PACK', 'GROSS_AMOUNT']].head())

,DATE,GROUP,WH,ITEM_CODE,SKU_DESCRIPTION,QTY_PACK,GROSS_AMOUNT
0,2025-01-02,INDOMARET,CEDIS MT SURABAYA,500189,BIG COLA 3100ML x6,20.0,2635913.20
1,2025-01-02,INDOMARET,CEDIS MT SURABAYA,500776,BIG STRAWBERRY 3100ML x6,22.0,2899504.52
2,2025-01-02,ALFAMIDI,CEDIS MT SURABAYA,522636,BIG STRAWBERRY PET NO RETORNAB,5.0,275482.60
3,2025-01-02,ALFAMIDI,CEDIS MT SURABAYA,523418,BIG SBRRY NRP 1625 ML 6,84.0,6016480.68
4,2025-01-02,ALFAMIDI,CEDIS MT SURABAYA,500189,BIG COLA 3100ML x6,13.0,1674398.70


## 5. Normalize promo calendar
Channel tidak membatasi scope. Promo diubah menjadi Item Code × Date; overlap mekanisme menjadi MULTIPLE_PROMO.


In [34]:
promo_source, promo_official, promo_episodes, promo_effect = (
    fp.load_clean_promo(PROMO_FILE)
)

{
    "source_rows": len(promo_source),
    "official_daily_rows": len(promo_official),
    "episodes": len(promo_episodes),
    "effect_rows": len(promo_effect),
}

{'source_rows': 1463,
 'official_daily_rows': 3506,
 'episodes': 91,
 'effect_rows': 3660}

### Sampel Kalender Promo (Raw Input)
Ini adalah data dari form promo yang diunggah oleh user sebelum diproses menjadi time-series.

In [35]:
# Memperbaiki tampilan data promo dengan mengecek kolom yang tersedia
# Berdasarkan state kernel, kolom yang tersedia kemungkinan adalah DISCOUNT_MAX atau nominal diskon lainnya

# Menampilkan kolom yang tersedia untuk verifikasi
print("Kolom tersedia di promo_source:", promo_source.columns.tolist())

# Menampilkan data dengan kolom yang pasti ada
display(promo_source[['ITEM_CODE', 'PROMO_CATEGORY', 'START_DATE', 'END_DATE']].head())

Kolom tersedia di promo_source: ['Id', 'Start time', 'Completion time', 'Email', 'Name', 'Channel', 'SKU', 'Mulai', 'Berakhir', 'Potongan Harga', 'Eksposure/Nama Promo', 'Biaya', 'START_DATE', 'END_DATE', 'ITEM_CODE', 'PROMO_CATEGORY', 'DISCOUNT_NUMERIC']


,ITEM_CODE,PROMO_CATEGORY,START_DATE,END_DATE
0,500040,CUT_PRICE,2025-10-09,2025-10-22
1,500040,CUT_PRICE,2025-10-23,2025-11-05
2,500040,CUT_PRICE,2025-10-03,2025-10-05
3,500040,CUT_PRICE,2025-10-17,2025-10-19
4,500040,CUT_PRICE,2025-10-24,2025-10-26


In [36]:
# Mengekstrak kategori promo unik dari data source
unique_promos = promo_source['PROMO_CATEGORY'].unique()
promo_table = pd.DataFrame(unique_promos, columns=['Unique Promo Category'])

print(f"Ditemukan {len(promo_table)} kategori promo unik.")
display(promo_table)

Ditemukan 3 kategori promo unik.


,Unique Promo Category
0,CUT_PRICE
1,SPECIAL_PRICE
2,FREE_GOODS


## 6. Build complete daily panel


In [37]:
panel, actual_cutoff = fp.build_daily_panel(mt, product_dim, promo_effect)
{'rows': len(panel), 'cutoff': actual_cutoff, 'series': panel[['GROUP','WH','ITEM_CODE']].drop_duplicates().shape[0]}


{'rows': 248875, 'cutoff': Timestamp('2026-08-12 00:00:00'), 'series': 720}

## 7. De-promote historical sales
Actual tetap disimpan. Pada promo-affected dates, baseline memakai nilai yang lebih rendah antara actual dan counterfactual no-promo agar proses de-promotion tidak menciptakan volume historis.


### **Glosarium Klasifikasi Demand (Syntetos-Boylan)**

Sistem mengelompokkan SKU berdasarkan pola penjualannya untuk menentukan model forecast terbaik:

| Metrik | Penjelasan |
| :--- | :--- |
| **ADI** | *Average Demand Interval*. Mengukur interval kemunculan demand. Nilai **> 1.32** dianggap *Intermittent* (jarang). |
| **CV²** | *Coefficient of Variation Squared*. Mengukur variabilitas jumlah. Nilai **> 0.49** dianggap *Volatile* (gejolak tinggi). |
| **ACTIVE_DAYS** | Jumlah hari unik di mana terjadi penjualan dalam periode histori. |

**Kategori:**
*   **STABLE**: Demand rutin dengan jumlah stabil.
*   **VOLATILE**: Demand rutin tapi jumlahnya naik-turun tajam.
*   **INTERMITTENT**: Demand jarang muncul tapi jumlahnya relatif stabil saat ada.
*   **SPARSE_NEW**: Produk baru atau data terlalu sedikit untuk dihitung statistiknya.

In [38]:
baseline_history = fp.build_baseline_history(panel, actual_cutoff)
demand_class = fp.classify_demand(baseline_history)
demand_class['DEMAND_CLASS'].value_counts()


,count
DEMAND_CLASS,
SPARSE_NEW,465
INTERMITTENT,138
VOLATILE,104
STABLE,13


### Glosarium Evaluasi Model

| Metrik | Penjelasan |
| :--- | :--- |
| **SELECTED_METHOD** | Algoritma yang dipilih otomatis berdasarkan performa *backtest* terbaik. |
| **BACKTEST_WAPE** | *Weighted Average Percentage Error*. Semakin **kecil** nilainya, semakin akurat model tersebut dalam memprediksi data historis. |
| **NAIVE_7 / 14** | Menggunakan data 7 atau 14 hari terakhir sebagai prediksi masa depan. |
| **RECENT_28** | Menggunakan rata-rata bergerak dari 28 hari terakhir. |

### Kesimpulan Hubungan:

*   **SPARSE_NEW**: Biasanya didominasi oleh `RECENT_28` karena sistem butuh rata-rata jangka panjang untuk menutupi banyaknya hari tanpa penjualan.
*   **STABLE**: Jika ada, biasanya akan menggunakan `WEEKDAY` atau `NAIVE_7` karena pola mingguannya sangat konsisten.
*   **VOLATILE**: Seringkali memiliki **WAPE** tertinggi. Jika error terlalu besar di kategori ini, kita biasanya menyarankan pengecekan manual atau penambahan faktor eksternal (seperti event khusus).

## 8. Rolling 14-day backtest and model selection


In [39]:
backtest_detail, backtest_summary, model_selection = fp.rolling_backtest(baseline_history, demand_class, actual_cutoff)
model_selection.groupby('SELECTED_METHOD').agg(series=('ITEM_CODE','size'), median_wape=('BACKTEST_WAPE','median'))


,series,median_wape
SELECTED_METHOD,,
NAIVE_7,30,1.015646
RECENT_28,668,1.768287
WEEKDAY_4,9,0.964839
WEEKDAY_8,13,1.045997


In [40]:
import pandas as pd
import plotly.express as px

# Menggabungkan data klasifikasi dengan hasil pemilihan model
relation_df = pd.merge(
    demand_class[['GROUP', 'WH', 'ITEM_CODE', 'DEMAND_CLASS']],
    model_selection[['GROUP', 'WH', 'ITEM_CODE', 'SELECTED_METHOD', 'BACKTEST_WAPE']],
    on=['GROUP', 'WH', 'ITEM_CODE']
)

# Membuat pivot table untuk melihat metode apa yang paling sering dipilih per kategori
summary_relation = relation_df.groupby(['DEMAND_CLASS', 'SELECTED_METHOD']).size().reset_index(name='COUNT')

fig = px.bar(summary_relation,
             x='DEMAND_CLASS',
             y='COUNT',
             color='SELECTED_METHOD',
             title='Hubungan Kategori Demand dengan Metode Forecast Terpilih',
             barmode='group',
             labels={'DEMAND_CLASS': 'Kategori Demand', 'COUNT': 'Jumlah SKU'})

fig.show()

# Menampilkan rata-rata error per kategori
error_summary = relation_df.groupby('DEMAND_CLASS')['BACKTEST_WAPE'].median().reset_index()
error_summary.columns = ['Kategori Demand', 'Median WAPE (Error)']
print("\nRingkasan Tingkat Kesulitan Forecast per Kategori (Semakin tinggi WAPE, semakin sulit diprediksi):")
display(error_summary)


Ringkasan Tingkat Kesulitan Forecast per Kategori (Semakin tinggi WAPE, semakin sulit diprediksi):


,Kategori Demand,Median WAPE (Error)
0,INTERMITTENT,1.000000
1,SPARSE_NEW,1.000000
2,STABLE,0.815084
3,VOLATILE,1.239692


In [41]:
# Mengecek granularity perhitungan
grain_summary = demand_class.groupby(['GROUP', 'WH']).agg(total_skus=('ITEM_CODE', 'nunique')).reset_index()

print(f"Total kombinasi Group-WH yang dihitung: {len(grain_summary)}")
print(f"Total unique series (Group-WH-SKU): {len(demand_class)}")

print("\nContoh sebaran SKU per Region & Channel:")
display(grain_summary.head(10))

Total kombinasi Group-WH yang dihitung: 89
Total unique series (Group-WH-SKU): 720

Contoh sebaran SKU per Region & Channel:


,GROUP,WH,total_skus
0,AEON,CEDIS MT SURABAYA,10
1,AEON,FACTORY,10
2,AEON,LOGISTIC PROVIDER BEKASI,10
3,AEON,LOGISTIC PROVIDER BOGOR,10
4,ALFAMART,CEDIS MT SURABAYA,8
5,ALFAMART,FACTORY,8
6,ALFAMART,LOGISTIC PROVIDER BALI,1
7,ALFAMART,LOGISTIC PROVIDER BANDUNG,3
8,ALFAMART,LOGISTIC PROVIDER BEKASI,7
9,ALFAMART,LOGISTIC PROVIDER BOGOR,7


## 9. Historical promo uplift
Uplift dihitung sebagai actual / counterfactual − 1 pada event window. Fallback: item → format → brand → category → global.


In [42]:
uplift_events, uplift_summaries = fp.build_uplift_table(baseline_history, product_dim)
uplift_events[['ITEM_CODE','PROMO_EFFECT_CATEGORY','EVENT_START','EVENT_END','ACTUAL_QTY','COUNTERFACTUAL_QTY','UPLIFT_RATE']].head()


,ITEM_CODE,PROMO_EFFECT_CATEGORY,EVENT_START,EVENT_END,ACTUAL_QTY,COUNTERFACTUAL_QTY,UPLIFT_RATE
0,500040,CUT_PRICE,2025-09-26,2025-11-13,10970.0,467.675000,2.0
1,500040,MULTIPLE_PROMO,2025-11-14,2025-12-03,565.0,105.083333,2.0
2,500040,CUT_PRICE,2025-12-04,2026-04-12,28255.0,3110.016667,2.0
3,500040,MULTIPLE_PROMO,2026-04-13,2026-05-03,1167.0,259.004167,2.0
4,500040,CUT_PRICE,2026-05-04,2026-08-12,3374.0,603.591667,2.0


In [43]:
# Menampilkan ringkasan uplift berdasarkan kategori promo (Global Category Level)
display(uplift_summaries['category'])



,PROMO_EFFECT_CATEGORY,EVENT_COUNT,MEDIAN_UPLIFT,MEAN_UPLIFT,LEVEL
0,CUT_PRICE,31,2.000000,1.449893,CATEGORY
1,MULTIPLE_PROMO,29,1.592074,1.294120,CATEGORY
2,SPECIAL_PRICE,3,0.000000,0.000000,CATEGORY


### **Glosarium Metrik Uplift**

Berikut adalah penjelasan kolom pada tabel ringkasan uplift:

| Kolom | Penjelasan |
| :--- | :--- |
| **EVENT_COUNT** | Jumlah total kejadian promo historis yang ditemukan untuk kategori tersebut. |
| **MEDIAN_UPLIFT** | Nilai tengah kenaikan penjualan. Contoh: `0.5` berarti kenaikan **50%** di atas baseline. |
| **MEAN_UPLIFT** | Rata-rata aritmatika dari kenaikan penjualan. |

> **Catatan:** Sistem menggunakan **Median** sebagai fallback utama karena lebih tahan terhadap *outlier* (data ekstrem) dibanding Mean.

## 10. Daily forecast and weekly aggregation


In [44]:
forecast_daily, forecast_weekly_group, forecast_weekly = fp.generate_forecast(panel, baseline_history, model_selection, demand_class, product_dim, uplift_summaries, actual_cutoff)
forecast_weekly.groupby('WEEK_START')[['BASELINE_PACK','PROMO_UPLIFT_PACK','FINAL_FC_PACK']].sum()


,BASELINE_PACK,PROMO_UPLIFT_PACK,FINAL_FC_PACK
WEEK_START,,,
2026-08-17,17731,15030,32761
2026-08-24,17608,11249,28857


In [53]:
import plotly.graph_objects as go

# Agregasi mingguan untuk melihat total koridor ketidakpastian
fc_summary = forecast_weekly.groupby('WEEK_START').agg({
    'FINAL_FC_PACK': 'sum',
    'LOW_FC_PACK': 'sum',
    'HIGH_FC_PACK': 'sum'
}).reset_index()

fig = go.Figure()

# 1. Area Ketidakpastian (Confidence Interval Range)
fig.add_trace(go.Scatter(
    x=pd.concat([fc_summary['WEEK_START'], fc_summary['WEEK_START'][::-1]]),
    y=pd.concat([fc_summary['HIGH_FC_PACK'], fc_summary['LOW_FC_PACK'][::-1]]),
    fill='toself',
    fillcolor='rgba(255, 0, 0, 0.2)',
    line=dict(color='rgba(255, 255, 255, 0)'),
    hoverinfo="skip",
    showlegend=True,
    name='Uncertainty Range (High-Low)'
))

# 2. Garis Forecast Utama (Point Forecast)
fig.add_trace(go.Scatter(
    x=fc_summary['WEEK_START'],
    y=fc_summary['FINAL_FC_PACK'],
    mode='lines+markers+text',
    text=[f"{x:,.0f}" for x in fc_summary['FINAL_FC_PACK']],
    textposition="top center",
    name='Final Forecast (Point)',
    line=dict(color='red', width=3)
))

fig.update_layout(
    title='Forecast Uncertainty Range (Total Modern Trade)',
    xaxis_title='Week Start',
    yaxis_title='Total Packs',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [62]:
import plotly.graph_objects as go
import pandas as pd

# Memastikan data terbaru digunakan
hist_df = baseline_history.copy()
hist_df['DATE'] = pd.to_datetime(hist_df['DATE'])
hist_df['WEEK_START'] = hist_df['DATE'] - pd.to_timedelta(hist_df['DATE'].dt.weekday, unit='D')

# Agregasi mingguan menggunakan kolom baseline yang sudah disinkronkan
hist_weekly = hist_df.groupby('WEEK_START').agg({
    'QTY_PACK': 'sum',
    'BASELINE_PACK': 'sum'
}).reset_index()

fig = go.Figure()

# Plot Historis: Actual vs Baseline
fig.add_trace(go.Scatter(x=hist_weekly['WEEK_START'], y=hist_weekly['QTY_PACK'], mode='lines', name='Actual (History)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=hist_weekly['WEEK_START'], y=hist_weekly['BASELINE_PACK'], mode='lines', name='Baseline (Cleaned)', line=dict(color='lightblue', dash='dash')))

# Plot Forecast
fig.add_trace(go.Scatter(x=forecast_weekly['WEEK_START'], y=forecast_weekly['BASELINE_PACK'], mode='lines+markers', name='Forecast Baseline', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=forecast_weekly['WEEK_START'], y=forecast_weekly['FINAL_FC_PACK'], mode='lines+markers', name='Forecast + Promo', line=dict(color='red', width=3)))

fig.update_layout(title='Weekly Sales Performance: Perbaikan Visualisasi Baseline', xaxis_title='Week Start', yaxis_title='Qty Packs', hovermode='x unified', template='plotly_white')
fig.show()

In [59]:
# Menentukan kolom baseline yang tersedia di dataframe
b_col = 'BASELINE_PACK' if 'BASELINE_PACK' in baseline_history.columns else ('BASELINE_QTY' if 'BASELINE_QTY' in baseline_history.columns else 'QTY_PACK')

# Mengecek statistik de-promosi di data historis
promo_rows = baseline_history[baseline_history['PROMO_AFFECTED'] == 1]
diff_rows = baseline_history[baseline_history['QTY_PACK'] != baseline_history[b_col]]

print(f"Menggunakan kolom baseline: {b_col}")
print(f"Total Baris Data Historis: {len(baseline_history)}")
print(f"Baris terdeteksi Promo   : {len(promo_rows)} ({len(promo_rows)/len(baseline_history)*100:.2f}%)")
print(f"Baris yang Baseline-nya berbeda dengan Actual: {len(diff_rows)}")

if len(promo_rows) > 0 and len(diff_rows) == 0:
    print("\nKESIMPULAN: Promo terdeteksi, tapi Baseline tetap sama dengan Actual.")
    print("Ini terjadi karena penjualan saat promo TIDAK LEBIH TINGGI dari counterfactual (rata-rata non-promo).")
    print("Sistem tidak akan menurunkan baseline jika tidak ada lonjakan signifikan.")

# Menampilkan sampel data promo
if len(promo_rows) > 0:
    print("\nSampel Data Promo (Cek Kolom QTY vs Baseline Terdeteksi):")
    cols_to_show = ['DATE', 'ITEM_CODE', 'QTY_PACK', b_col, 'PROMO_EFFECT_CATEGORY']
    display(promo_rows[cols_to_show].head(10))

Menggunakan kolom baseline: BASELINE_PACK
Total Baris Data Historis: 235915
Baris terdeteksi Promo   : 102561 (43.47%)
Baris yang Baseline-nya berbeda dengan Actual: 6844

Sampel Data Promo (Cek Kolom QTY vs Baseline Terdeteksi):


,DATE,ITEM_CODE,QTY_PACK,BASELINE_PACK,PROMO_EFFECT_CATEGORY
0,2025-11-22,500040,1.0,1.0,MULTIPLE_PROMO
1,2025-11-23,500040,0.0,0.0,MULTIPLE_PROMO
2,2025-11-24,500040,0.0,0.0,MULTIPLE_PROMO
3,2025-11-25,500040,0.0,0.0,MULTIPLE_PROMO
4,2025-11-26,500040,0.0,0.0,MULTIPLE_PROMO
5,2025-11-27,500040,0.0,0.0,MULTIPLE_PROMO
6,2025-11-28,500040,0.0,0.0,MULTIPLE_PROMO
7,2025-11-29,500040,0.0,0.0,MULTIPLE_PROMO
8,2025-11-30,500040,0.0,0.0,MULTIPLE_PROMO
9,2025-12-01,500040,0.0,0.0,MULTIPLE_PROMO


### **Analisis Efektivitas Promo (Actual vs Counterfactual)**

Jika Baseline menumpuk dengan Actual, artinya sistem menilai performa penjualan saat promo tersebut **di bawah atau sama dengan** rata-rata hari tanpa promo. Mari kita visualisasikan distribusinya untuk melihat apakah ada *uplift* yang terdeteksi secara statistik.

In [56]:
import plotly.express as px

# Membandingkan distribusi QTY pada hari Promo vs Non-Promo
fig_dist = px.box(baseline_history,
                 x='PROMO_AFFECTED',
                 y='QTY_PACK',
                 color='PROMO_AFFECTED',
                 points="all",
                 title='Distribusi Penjualan: Non-Promo (0) vs Promo-Affected (1)',
                 labels={'PROMO_AFFECTED': 'Status Promo', 'QTY_PACK': 'Quantity (Packs)'},
                 category_orders={"PROMO_AFFECTED": [0, 1]})

fig_dist.update_layout(template='plotly_white')
fig_dist.show()

# Menghitung rata-rata sederhana
summary_stats = baseline_history.groupby('PROMO_AFFECTED')['QTY_PACK'].mean().reset_index()
summary_stats.columns = ['Status Promo', 'Rata-rata QTY per Hari']
display(summary_stats)

,Status Promo,Rata-rata QTY per Hari
0,0,11.106229
1,1,22.983649


### **Sinkronisasi Kolom Baseline**

Saya akan memastikan kolom `BASELINE_PACK` tersedia secara konsisten di dataframe `baseline_history` agar grafik perbandingan di atas berfungsi dengan benar.

In [57]:
# Memastikan kolom BASELINE_PACK ada di history untuk audit visual
if 'BASELINE_HISTORY_QTY' in baseline_history.columns:
    baseline_history['BASELINE_PACK'] = baseline_history['BASELINE_HISTORY_QTY']
elif 'COUNTERFACTUAL_QTY' in baseline_history.columns:
    # Logika sistem: Baseline = MIN(Actual, Counterfactual)
    baseline_history['BASELINE_PACK'] = baseline_history[['QTY_PACK', 'COUNTERFACTUAL_QTY']].min(axis=1)

print("✅ Kolom 'BASELINE_PACK' telah disinkronkan. Silakan jalankan ulang cell grafik 'Weekly Sales Performance' di atas.")

✅ Kolom 'BASELINE_PACK' telah disinkronkan. Silakan jalankan ulang cell grafik 'Weekly Sales Performance' di atas.


## 11. QA, reconciliation, and exports


In [51]:
qc, qc_summary, model_benchmark = fp.build_qc(raw, clean, mt, customer_master, promo_source, promo_official, product_dim, panel, baseline_history, backtest_summary, forecast_daily, forecast_weekly, ncc_qc)
qc


,CHECK,ACTUAL,EXPECTED,PASS
0,Actual cutoff,2026-08-12,2026-08-12,True
1,Modern rows mapped to Group,26068,26068,True
2,Negative clean MT rows,0,0,True
3,Promo item codes mapped to sales,12,12,True
4,Forecast week count,2,2,True
5,Weekly final reconciliation,0,0,True
6,Forecast negative rows,0,0,True


In [52]:
results = fp.export_all(locals())
results


{'xlsx': '/content/drive/MyDrive/MT_Forecast/output/MT_Weekly_Forecast_2026-08-17_2026-08-24.xlsx',
 'html': '/content/drive/MyDrive/MT_Forecast/output/MT_Weekly_Forecast_2026-08-17_2026-08-24_Plotly.html',
 'ipynb': 'C:\\Users\\didik.priyo.id\\Documents\\Codex\\2026-08-17\\ini/outputs/MT_Weekly_Forecast_2026-08-17_2026-08-24.ipynb',
 'python': 'C:\\Users\\didik.priyo.id\\Documents\\Codex\\2026-08-17\\ini/outputs/MT_Weekly_Forecast_Pipeline.py'}

## 12. Deep Dive: Analisis De-promosi & Troubleshooting

Bagian ini digunakan untuk memverifikasi logika de-promosi jika ditemukan kejanggalan pada grafik (misal: Baseline menumpuk dengan Actual).

### 12.1 Sinkronisasi Data Baseline
Memastikan kolom audit tersedia di seluruh dataset historis.

In [60]:
# Memastikan kolom audit tersedia
if 'BASELINE_HISTORY_QTY' in baseline_history.columns:
    baseline_history['BASELINE_PACK'] = baseline_history['BASELINE_HISTORY_QTY']
elif 'COUNTERFACTUAL_QTY' in baseline_history.columns:
    baseline_history['BASELINE_PACK'] = baseline_history[['QTY_PACK', 'COUNTERFACTUAL_QTY']].min(axis=1)

print("✅ Status: Kolom BASELINE_PACK siap untuk divisualisasikan.")

✅ Status: Kolom BASELINE_PACK siap untuk divisualisasikan.


### 12.2 Uji Statistik: Promo vs Non-Promo
Visualisasi untuk membuktikan apakah promo benar-benar menghasilkan lonjakan volume secara signifikan atau tidak.

In [61]:
import plotly.express as px

# 1. Boxplot Distribusi
fig_check = px.box(baseline_history,
                  x='PROMO_AFFECTED',
                  y='QTY_PACK',
                  color='PROMO_AFFECTED',
                  points="outliers",
                  title='Perbandingan Distribusi Penjualan: Normal (0) vs Promo (1)',
                  labels={'PROMO_AFFECTED': 'Status Promo', 'QTY_PACK': 'Qty Packs'})
fig_check.update_layout(template='plotly_white')
fig_check.show()

# 2. Ringkasan Statistik
summary = baseline_history.groupby('PROMO_AFFECTED').agg({
    'QTY_PACK': ['count', 'mean', 'median', 'std']
}).reset_index()
display(summary)

PROMO_AFFECTED QTY_PACK                              
                    count       mean median         std
0              0   133354  11.106229    0.0  268.648517
1              1   102561  22.983649    0.0  327.523078

### 12.3 Kesimpulan Diagnosa
Jika rata-rata (mean) atau median pada status Promo (1) tidak jauh lebih tinggi dari status Normal (0), maka sistem secara otomatis akan menetapkan **Baseline = Actual**. Ini bertujuan agar model tidak belajar dari 'lonjakan semu' yang sebenarnya tidak ada.

In [ ]:
# Cek 1 SKU yang memiliki perbedaan paling besar antara Actual dan Baseline
audit_sku = baseline_history[baseline_history['QTY_PACK'] != baseline_history['BASELINE_PACK']].copy()
if not audit_sku.empty:
    example_item = audit_sku['ITEM_CODE'].iloc[0]
    sample_data = baseline_history[baseline_history['ITEM_CODE'] == example_item].sort_values('DATE').tail(30)

    print(f"🔍 AUDIT SKU: {example_item}")
    display(sample_data[['DATE', 'QTY_PACK', 'BASELINE_PACK', 'PROMO_AFFECTED', 'PROMO_EFFECT_CATEGORY']])

    fig_audit = px.line(sample_data, x='DATE', y=['QTY_PACK', 'BASELINE_PACK'],
                       title=f'Audit De-promosi: Item {example_item}',
                       labels={'value': 'Packs', 'variable': 'Type'})
    fig_audit.show()
else:
    print("⚠️ Tidak ditemukan perbedaan antara Actual dan Baseline di level record.")